### Introduction to Machine Learning
Jesús Urtasun Elizari: MRC LMS 2026

**Chapter 2: Simple mathematical intuitions. Exercise 1**

**Loss functions**

A loss function measures, with a single number, how far the predictions of a model are from the true values.
Training a model means finding the parameters that make this number as small as possible. We implement three common losses by hand and check them with TensorFlow, PyTorch and SciPy.

**The maths**

Mean squared error (MSE), for continuous values such as gene expression: the average of the squared differences between true values $y_i$ and predictions $\hat{y}_i$:

$$\text{MSE} = \frac{1}{N} \sum_{i=1}^{N} \left( y_i - \hat{y}_i \right)^2$$

Chi-squared, for counts: the squared difference between observed $O_i$ and expected $E_i$ counts, divided by the expected count, so each difference is compared to the size of what we expected:

$$\chi^2 = \sum_{i} \frac{(O_i - E_i)^2}{E_i}$$

Binary cross-entropy, for yes / no labels $y_i$ (1 or 0) and predicted probabilities $p_i$: it is small when the model gives a high probability to the correct answer, and very large when it is confidently wrong:

$$\text{BCE} = -\frac{1}{N} \sum_{i=1}^{N} \left[ y_i \log(p_i) + (1 - y_i) \log(1 - p_i) \right]$$

In [ ]:
# Import libraries
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
import torch
from scipy import stats

# Check installation
print("All libraries imported successfully")
print(np.__version__)
print(tf.__version__)
print(torch.__version__)

In [ ]:
# Example data for MSE: true and predicted gene expression
y_true = np.array([1.0, 2.0, 3.0])
y_pred = np.array([1.1, 1.9, 3.2])
print("y_true:", y_true)
print("y_pred:", y_pred)
print("Shape:", y_true.shape)
print("Format: ", type(y_true))

In [ ]:
# Example data for chi-squared: number of cells of three types
# Observed in the experiment and expected from a model (same total, 50 cells)
observed = np.array([18, 25, 7])
expected = np.array([20, 20, 10])
print("Observed:", observed)
print("Expected:", expected)

In [ ]:
# Example data for binary cross-entropy: is the cell a tumour cell? (1 = yes, 0 = no)
# The model gives a probability of being a tumour cell
labels = np.array([1, 0, 1])
probs = np.array([0.9, 0.2, 0.8])
print("Labels:       ", labels)
print("Probabilities:", probs)

In [ ]:
# Manual implementation

# Mean squared error
def mse_manual(y_true, y_pred):
    return np.mean((y_true - y_pred)**2)

# Chi-squared
def chi2_manual(observed, expected):
    return np.sum((observed - expected)**2 / expected)

# Binary cross-entropy
def bce_manual(labels, probs):
    return -np.mean(labels * np.log(probs) + (1 - labels) * np.log(1 - probs))

mse_m = mse_manual(y_true, y_pred)
chi2_m = chi2_manual(observed, expected)
bce_m = bce_manual(labels, probs)
print("\nManual implementation:")
print("MSE:", mse_m)
print("Chi-squared:", chi2_m)
print("Binary cross-entropy:", bce_m)

In [ ]:
# Plot each loss as a function of one prediction
fig, axes = plt.subplots(1, 3, figsize = (15, 4))

# MSE for one point with true value 2: a parabola, smallest when the prediction is 2
pred = np.linspace(0, 4, 200)
axes[0].plot(pred, (2 - pred)**2)
axes[0].set_title("MSE (true value = 2)")
axes[0].set_xlabel("prediction")
axes[0].set_ylabel("loss")
axes[0].grid(True)

# Chi-squared for one category with 20 observed cells: smallest when we expect 20
exp_counts = np.linspace(5, 40, 200)
axes[1].plot(exp_counts, (20 - exp_counts)**2 / exp_counts)
axes[1].set_title("Chi-squared (observed = 20)")
axes[1].set_xlabel("expected count")
axes[1].set_ylabel("loss")
axes[1].grid(True)

# Binary cross-entropy for one tumour cell (label 1): grows fast when p goes to 0
p = np.linspace(0.01, 0.99, 200)
axes[2].plot(p, -np.log(p))
axes[2].set_title("Binary cross-entropy (label = 1)")
axes[2].set_xlabel("predicted probability")
axes[2].set_ylabel("loss")
axes[2].grid(True)

plt.tight_layout()
plt.show()

In [ ]:
# TensorFlow implementation
mse_tf = tf.keras.losses.MeanSquaredError()(y_true, y_pred).numpy()
bce_tf = tf.keras.losses.BinaryCrossentropy()(labels, probs).numpy()
print("\nTensorFlow implementation:")
print("MSE:", mse_tf)
print("Binary cross-entropy:", bce_tf)

In [ ]:
# PyTorch implementation

# Convert input to required format
y_true_torch = torch.tensor(y_true, dtype = torch.float32)
y_pred_torch = torch.tensor(y_pred, dtype = torch.float32)
labels_torch = torch.tensor(labels, dtype = torch.float32)
probs_torch = torch.tensor(probs, dtype = torch.float32)

mse_torch = torch.nn.functional.mse_loss(y_pred_torch, y_true_torch).item()
bce_torch = torch.nn.functional.binary_cross_entropy(probs_torch, labels_torch).item()
print("\nPyTorch implementation:")
print("MSE:", mse_torch)
print("Binary cross-entropy:", bce_torch)

In [ ]:
# SciPy implementation of chi-squared
chi2_scipy = stats.chisquare(observed, expected).statistic
print("\nSciPy implementation:")
print("Chi-squared:", chi2_scipy)

In [ ]:
# Precision check (rtol = 1e-5)

print("\nPrecision check (Manual vs TensorFlow):")
print("MSE close: ", np.allclose(mse_m, mse_tf))
print("Binary cross-entropy close: ", np.allclose(bce_m, bce_tf))

print("\nCross-check (Manual vs PyTorch):")
print("MSE close: ", np.allclose(mse_m, mse_torch))
print("Binary cross-entropy close: ", np.allclose(bce_m, bce_torch))

print("\nCross-check (Manual vs SciPy):")
print("Chi-squared close: ", np.allclose(chi2_m, chi2_scipy))